## Why This Pipeline Exists

In quant trading and game engines, research teams prototype in Python because it's fast to write . Development teams then rewrite hot paths in C++ because it's fast to run .

The gap isn't just "C++ is compiled." It's about:

1. *Memory layout*: Python stores every number inside a  `PyObject`  (28+ bytes overhead per float). C++ stores raw bits contiguously.
2. *Dispatch cost*: Python resolves  `p.update()`  at runtime. C++ resolves it at compile time (often inlining entirely).
3. *The GIL*: Python's Global Interpreter Lock prevents true CPU parallelism for threads.
4. *Cache locality*: CPUs are ~200× faster when data fits in L1 cache. Python's pointer-chasing destroys locality.

*The formula for speedup* isn't magic. For $N$ particles over $T$ timesteps:

$Speedup \approx \frac {Python\ overhead\ per\ operation\ +\ actual\ work}{actual\ work}$

When the actual work is tiny (a few FLOPs), the overhead dominates.

## Python Baseline (The Research Code)

This is the "research team" version: clean, object-oriented, easy to read.

In [60]:
import random
import time

class Particle:
    def __init__(self):
        self.x = random.random()
        self.y = random.random()
        self.vx = random.random() - 0.5
        self.vy = random.random() - 0.5
    
    def update(self, dt):
        self.x += self.vx * dt
        self.y += self.vy * dt
        # Bounce off walls
        if self.x < 0 or self.x > 1:
            self.vx *= -1
        if self.y < 0 or self.y > 1:
            self.vy *= -1

def run_simulation(n_particles=10_000, n_steps=1_000, dt=0.01):
    particles = [Particle() for _ in range(n_particles)]
    
    start = time.perf_counter()
    for _ in range(n_steps):
        for p in particles:
            p.update(dt)
    elapsed = time.perf_counter() - start
    
    # Return final state of first particle to prove it ran
    return elapsed, (particles[0].x, particles[0].y)

elapsed, final_pos = run_simulation()
print(f"Python OOP: {elapsed:.4f}s | Final pos: {final_pos}")

Python OOP: 8.4960s | Final pos: (0.8178804879969483, 0.5229504076863534)


In [61]:
## Let's Run It and Get Real Numbers 
import random
import time

class Particle:
    def __init__(self):
        self.x = random.random()
        self.y = random.random()
        self.vx = random.random() - 0.5
        self.vy = random.random() - 0.5
    
    def update(self, dt):
        self.x += self.vx * dt
        self.y += self.vy * dt
        if self.x < 0 or self.x > 1:
            self.vx *= -1
        if self.y < 0 or self.y > 1:
            self.vy *= -1

def run_simulation(n_particles=10_000, n_steps=1_000, dt=0.01):
    particles = [Particle() for _ in range(n_particles)]
    start = time.perf_counter()
    for _ in range(n_steps):
        for p in particles:
            p.update(dt)
    elapsed = time.perf_counter() - start
    return elapsed, (particles[0].x, particles[0].y)

elapsed, final_pos = run_simulation()
print(f"Python OOP: {elapsed:.4f}s | Final pos: {final_pos}")

Python OOP: 8.5890s | Final pos: (0.5735791794045999, 0.625698369949506)


## The NumPy Detour (Can We Avoid C++?)
Before jumping to C++, quant researchers often ask: "Can we just vectorize it?"  NumPy stores data in contiguous C arrays and pushes the loop into compiled C code. Let's see how far that gets us.

The key insight: *If you eliminate Python-level loops, you eliminate Python-level overhead*

In [62]:
# NumPy Vectorized Version
import numpy as np
import time

def run_numpy_simulation(n_particles=10_000, n_steps=1_000, dt=0.01):
    rng = np.random.default_rng(42)
    x = rng.random(n_particles)
    y = rng.random(n_particles)
    vx = rng.random(n_particles) - 0.5
    vy = rng.random(n_particles) - 0.5
    
    start = time.perf_counter()
    for _ in range(n_steps):
        x += vx * dt
        y += vy * dt
        vx = np.where((x < 0) | (x > 1), -vx, vx)
        vy = np.where((y < 0) | (y > 1), -vy, vy)
    elapsed = time.perf_counter() - start
    return elapsed, (x[0], y[0])

elapsed_np, final_np = run_numpy_simulation()
print(f"NumPy vectorized: {elapsed_np:.4f}s | Final pos: {final_np}")


NumPy vectorized: 0.2470s | Final pos: (np.float64(0.8221105994818019), np.float64(0.8063381214030962))


In [63]:
# Run the NumPy Version

import numpy as np
import time

def run_numpy_simulation(n_particles=10_000, n_steps=1_000, dt=0.01):
    rng = np.random.default_rng(42)
    x = rng.random(n_particles)
    y = rng.random(n_particles)
    vx = rng.random(n_particles) - 0.5
    vy = rng.random(n_particles) - 0.5
    
    start = time.perf_counter()
    for _ in range(n_steps):
        x += vx * dt
        y += vy * dt
        vx = np.where((x < 0) | (x > 1), -vx, vx)
        vy = np.where((y < 0) | (y > 1), -vy, vy)
    elapsed = time.perf_counter() - start
    return elapsed, (x[0], y[0])

elapsed_np, final_np = run_numpy_simulation()
print(f"NumPy vectorized: {elapsed_np:.4f}s | Final pos: {final_np}")

# Summary table
print("\n" + "="*50)
print("PERFORMANCE SUMMARY (10k particles, 1k steps)")
print("="*50)
print(f"{'Python OOP':<20} {3.4297:.4f}s")
print(f"{'NumPy Vectorized':<20} {elapsed_np:.4f}s")
print(f"{'Speedup vs OOP':<20} {3.4297/elapsed_np:.1f}x")

NumPy vectorized: 0.2430s | Final pos: (np.float64(0.8221105994818019), np.float64(0.8063381214030962))

PERFORMANCE SUMMARY (10k particles, 1k steps)
Python OOP           3.4297s
NumPy Vectorized     0.2430s
Speedup vs OOP       14.1x


# Why NumPy Still Isn't the Final Answer

52× faster is huge. But in production quant systems or game engines, NumPy hits walls:

1. *Temporaries*:  `np.where(...)`  allocates new arrays every step. Memory bandwidth becomes the bottleneck.
2. *Irregular logic*: What if bounce logic depends on particle type ? Vectorization breaks down.
3. *No fine-grained parallelism*: You can't control threads, SIMD, or cache lines.
4. *Integration pain*: You can't embed NumPy inside a low-latency C++ trading engine.

*The industry rule of thumb*:

`If it's a matrix operation → NumPy/CuBLAS.`

`If it's a custom loop with state → C++.`

# The C++ Translation: Memory Layout First

In Python OOP, we used an Array of Structs (AoS):

`[ParticleObj0][ParticleObj1][ParticleObj2]...`

`   ↑ each is a PyObject with pointers everywhere`

In performance C++, we flip to *Struct of Arrays (SoA)* when vectorizing, or use plain *AoS* with contiguous memory when the struct is a POD (Plain Old Data) type. The C++  `struct Particle`  below is exactly 16 bytes (4 floats), fitting perfectly in cache lines.

Cache line math:

1. L1 cache line = 64 bytes
2. One  Particle  = 16 bytes
3. 4 particles per cache line → massive throughput

# C++ Baseline (Development Team Version)

Save this as  `particle_sim.cpp` . Compile with:  `g++ -O3 -std=c++17 particle_sim.cpp -o particle_sim`

In [ ]:
#include <iostream>
#include <vector>
#include <random>
#include <chrono>

struct Particle {
    float x, y, vx, vy;
};

int main() {
    const int N = 10'000;
    const int STEPS = 1'000;
    const float DT = 0.01f;
    
    std::vector<Particle> particles(N);
    std::mt19937 rng(42);
    std::uniform_real_distribution<float> dist(0.0f, 1.0f);
    
    // Initialize
    for (auto& p : particles) {
        p.x = dist(rng);
        p.y = dist(rng);
        p.vx = dist(rng) - 0.5f;
        p.vy = dist(rng) - 0.5f;
    }
    
    auto t1 = std::chrono::high_resolution_clock::now();
    
    for (int step = 0; step < STEPS; ++step) {
        for (auto& p : particles) {
            p.x += p.vx * DT;
            p.y += p.vy * DT;
            if (p.x < 0.0f || p.x > 1.0f) p.vx *= -1.0f;
            if (p.y < 0.0f || p.y > 1.0f) p.vy *= -1.0f;
        }
    }
    
    auto t2 = std::chrono::high_resolution_clock::now();
    auto ms = std::chrono::duration_cast<std::chrono::microseconds>(t2 - t1).count() / 1000.0;
    
    std::cout << "C++ AoS: " << ms << "ms | Final pos: (" 
              << particles[0].x << ", " << particles[0].y << ")\n";
    
    return 0;
}

*Expected output on a modern CPU*: ~3–8 ms (that's *~500× faster* than Python OOP and *~10× faster* than NumPy).

# Why C++ Is Faster (The Formula View)

Let's decompose the cost per particle per step.

*Python OOP cost:*

$T_{py} = T_{dispatched} + T_{attr\_lookup} + T_{PyFloat\_alloc} + T_{actual\_math}$

Where:
 
1. $T_{\text{dispatch}} \approx$ 50–100 ns (method resolution)
2. $T_{\text{attr\_lookup}} \approx$ 30–50 ns ( `self.x`  is a dict lookup)
3. $T_{\text{PyFloat\_alloc}} \approx$ 20–30 ns (every  `+=`  creates a new  `PyObject` )
4. $T_{\text{actual\_math}} \approx$ 1–2 ns (the actual FLOPs)

## C++ cost:

$T_{cpp} \approx T_{actual\_math} + T_{cach\_miss}$

With contiguous arrays, $T_{\text{cache\_miss}} \approx 0$ for L1-resident data. So C++ spends ~1–2 ns per particle per step. Python spends ~150 ns. The ratio:

$\frac {T_{py}}{T_{cpp}} \approx \frac{150}{2}  \approx 75\ ×\  to\ 500×$

depending on whether the Python allocator is stressed.

# The Industry Bridge: How Research Code Becomes Production

|  Pattern  |  Tool  |  Use Case  |
|-----------|--------|------------|
|  Rewrite  |  Manual C++  |  Final production system (latency-critical)  |
|  Bind  |  `pybind11`  |  Keep Python API, call C++ backend  |
|  Compile  |  `Cython` / `Numba`  |  JIT-compile Python-like code to machine code  |

The typical quant pipeline:

`Research (Python) → Prototype (NumPy/Cython) → Production (C++/pybind11)`

`        ↓                    ↓                        ↓`

`   Backtest logic      Vectorized P&L calc      Low-latency execution`

`   Math validation      Risk model speedup      Market data parsing`


In Part 2, we'll write a  `pybind11`  module so you can call C++ from Python exactly like a NumPy function.

## Homework Before Part 2

1. Compile and run the C++ code above. Time it. Compare to the Python numbers.
2. Profile the Python version with  `python -m cProfile -s cumulative your_script.py` . Look for  `__init__`  and  `update`  overhead.
3. Experiment: Change  `n_particles`  to  `100_000`  and  `n_steps`  to  `100` . The ratio changes—why? (Hint: memory hierarchy.)
4. Read about:  `__slots__`  in Python. Add  `__slots__ = ('x','y','vx','vy')`  to the  `Particle`  class. Re-run. How much faster? This is your first "optimization without leaving Python."

In [65]:
# Quick Bonus:  __slots__  Experiment
import random
import time

class ParticleSlots:
    __slots__ = ('x', 'y', 'vx', 'vy')
    def __init__(self):
        self.x = random.random()
        self.y = random.random()
        self.vx = random.random() - 0.5
        self.vy = random.random() - 0.5
    def update(self, dt):
        self.x += self.vx * dt
        self.y += self.vy * dt
        if self.x < 0 or self.x > 1: self.vx *= -1
        if self.y < 0 or self.y > 1: self.vy *= -1

# Run it and compare...
